# Load CMS raw CSVs into lakehouse tables

Reads the CSV files in `Files/raw/<DATASET>/` of the default lakehouse and writes three Delta tables:

| Table | Source folder | One row per | Columns |
|---|---|---|---|
| `PSI90` | `raw/PSI` | hospital per CSV release | one column per patient safety indicator (`Measure Name`) |
| `HAI` | `raw/HAI` | hospital per CSV release | one column per `Measure ID-Measure Name` |
| `HCAHPS` | `raw/HCAHPS` | hospital per CSV release | one column per `HCAHPS Measure ID-HCAHPS Answer Description` for the star rating and linear mean score measures |
| `dim_date` | generated | calendar day | `Date`, `Year`, `MonthNo`, `Month`, `QuarterNo`, `Quarter`, `Day` |

Every table also has `REPORTYEAR` (year of the file's End Date) and `FACILITYYEARKEY` (`Facility ID` + `REPORTYEAR`), the key used to relate the tables in the semantic model.

Physical column names use only letters, digits and underscores (Delta/Spark restriction). The semantic model maps them back to the readable names.

In [ ]:
import re
from functools import reduce
from pyspark.sql import functions as F

RAW_ROOT = "Files/raw"  # relative to the default lakehouse


def sanitize(name):
    """Readable measure name -> physical Delta column name."""
    return re.sub(r"[^0-9A-Za-z]+", "_", name).strip("_")


def read_folder(folder, extra_cols):
    """Read every CSV of a raw sub-folder. Column names changed between CMS releases, so files are read one at a time."""
    paths = sorted(f.path for f in notebookutils.fs.ls(f"{RAW_ROOT}/{folder}") if f.name.lower().endswith(".csv"))
    print(f"{folder}: {len(paths)} files")
    frames = []
    for path in paths:
        df = spark.read.option("header", True).option("quote", '"').option("escape", '"').csv(path)
        if "City/Town" in df.columns:
            df = df.withColumnRenamed("City/Town", "City")
        needed = ["Facility ID", "Facility Name", "Address", "City", "State", "ZIP Code", "Start Date", "End Date"] + extra_cols
        frames.append(df.select(*needed).withColumn("_source_file", F.lit(path.split("/")[-1])))
    return reduce(lambda a, b: a.unionByName(b), frames)


def build_table(df, pivot_expr, value_expr, table_name, drop_null_values=False):
    """Long CMS measure rows -> one wide row per hospital per source file."""
    df = df.withColumn("_pivot", pivot_expr).withColumn("_value", value_expr)
    if drop_null_values:
        df = df.where(F.col("_value").isNotNull())
    keys = ["_source_file", "Facility ID"]

    pivot_values = [r[0] for r in df.select("_pivot").distinct().collect()]
    physical = {v: sanitize(v) for v in pivot_values}
    assert len(set(physical.values())) == len(physical), "column name collision after sanitizing"

    attrs = df.groupBy(*keys).agg(
        F.max("Facility Name").alias("facility_name"),
        F.max("Address").alias("ADDRESS"),
        F.max("City").alias("CITY"),
        F.max("State").alias("STATE"),
        F.max("ZIP Code").alias("zip_code"),
        F.max(F.to_date("Start Date", "MM/dd/yyyy")).alias("start_date"),
        F.max(F.to_date("End Date", "MM/dd/yyyy")).alias("end_date"),
    )
    wide = df.groupBy(*keys).pivot("_pivot", sorted(pivot_values)).agg(F.first("_value"))
    for original, new in physical.items():
        wide = wide.withColumnRenamed(original, new)

    out = (
        attrs.join(wide, keys, "left")
        .withColumnRenamed("Facility ID", "facility_id")
        .withColumn("REPORTYEAR", F.year("end_date").cast("double"))
        .withColumn("FACILITYYEARKEY", F.concat(F.col("facility_id"), F.year("end_date").cast("string")))
        .drop("_source_file")
    )
    out.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(table_name)
    print(f"{table_name}: {spark.table(table_name).count()} rows, {len(physical)} measure columns")
    return physical


## PSI90
Patient Safety Indicators. Only the current layout (`Measure ID` like `PSI_03`) is loaded. The oldest release uses a different naming scheme (`PSI-6` ids and plain-language names), so it stays in `raw/PSI` but is not loaded.

In [ ]:
psi = read_folder("PSI", ["Measure ID", "Measure Name", "Rate"]).where(F.col("Measure ID").rlike("^PSI_"))
psi_cols = build_table(psi, F.col("Measure Name"), F.expr("try_cast(Rate as double)"), "PSI90")

## HAI
Healthcare Associated Infections. `Score` is numeric or text such as `Not Available`; text becomes null.

In [ ]:
hai = read_folder("HAI", ["Measure ID", "Measure Name", "Score"])
hai_cols = build_table(
    hai,
    F.concat_ws("-", F.col("Measure ID"), F.col("Measure Name")),
    F.expr("try_cast(Score as double)"),
    "HAI",
)

## HCAHPS
Patient experience. Only the star rating and linear mean score measures are loaded. Star rating measures take the value from `Patient Survey Star Rating`, linear score measures from `HCAHPS Linear Mean Value`. Hospitals with no star or linear value in a release are left out.

In [ ]:
hcahps = read_folder(
    "HCAHPS",
    ["HCAHPS Measure ID", "HCAHPS Answer Description", "Patient Survey Star Rating", "HCAHPS Linear Mean Value"],
).where(F.col("HCAHPS Measure ID").rlike("LINEAR_SCORE|STAR_RATING"))

hcahps_cols = build_table(
    hcahps,
    F.concat_ws("-", F.col("HCAHPS Measure ID"), F.col("HCAHPS Answer Description")),
    F.when(
        F.col("HCAHPS Measure ID").contains("STAR_RATING"), F.expr("try_cast(`Patient Survey Star Rating` as double)")
    ).otherwise(F.expr("try_cast(`HCAHPS Linear Mean Value` as double)")),
    "HCAHPS",
    drop_null_values=True,
)

## dim_date
Calendar table covering every year in the three tables. The semantic model uses it for the six hidden date tables that relate each table's `start date` and `end date` to a date hierarchy (Direct Lake cannot use calculated date tables).

In [ ]:
bounds = [spark.table(t).agg(F.min("start_date"), F.max("end_date")).first() for t in ["PSI90", "HAI", "HCAHPS"]]
first_year = min(b[0].year for b in bounds)
last_year = max(b[1].year for b in bounds)

dim_date = (
    spark.sql(f"SELECT explode(sequence(to_date('{first_year}-01-01'), to_date('{last_year}-12-31'), interval 1 day)) AS Date")
    .withColumn("Year", F.year("Date"))
    .withColumn("MonthNo", F.month("Date"))
    .withColumn("Month", F.date_format("Date", "MMMM"))
    .withColumn("QuarterNo", F.ceil(F.col("MonthNo") / 3).cast("int"))
    .withColumn("Quarter", F.concat(F.lit("Qtr "), F.col("QuarterNo")))
    .withColumn("Day", F.dayofmonth("Date"))
)
dim_date.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("dim_date")
print(f"dim_date: {spark.table('dim_date').count()} rows ({first_year}-{last_year})")

## Check
Row counts per table and report year.

In [ ]:
for t in ["PSI90", "HAI", "HCAHPS"]:
    print(t)
    spark.table(t).groupBy("REPORTYEAR").count().orderBy("REPORTYEAR").show()